### Move lost source files to new zotero db attachments directory

My zotero DB lost track of some of its source pdf and attachment files.  Copy them from there to the right spot in the new attachments directory, if they wouldn't overwrite files already there, and if the current zotero db has citekeys that would match them.

In [ ]:
from icecream import ic
import pathlib as pl
from pyzotero import zotero
import pandas as pd
from IPython.display import display, HTML

import pathlib as pl
from icecream import ic
import sys

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2


zot = zotero.Zotero(rfw.zotero_library_id, rfw.zotero_library_type, rfw.zotero_api_key)
parentItems = zot.everything(zot.top())


In [21]:
zotDBparentCiteKeys = [rfw.get_citation_key(parent['data']) for parent in parentItems]
print(f'{len(zotDBparentCiteKeys)=}')
#old_source_dir = pl.Path(r"C:\Users\scott\OneDrive\share\ref\papers")
#old_source_dir = pl.Path(r"C:\Users\scott\OneDrive\share\ref\DOE_brainstorm\papers")
#old_source_dir = rfw.refdir / 'zotero_SAVE_240401'
#old_source_dir = pl.Path(r"C:\Users\scott\OneDrive\share\transfer\papers_old")
#old_source_dir = pl.Path(r"C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_notes_OLD_PARTIAL")
old_source_dir = pl.Path(r'c:/Users/scott/OneDrive/share/transfer/papers_old')
old_source_dir.exists()

len(zotDBparentCiteKeys)=1674


True

In [22]:
def build_file_df(filedir):
    files = [fNm for fNm in filedir.glob('*')]
    return pd.DataFrame(dict(fullPath=files, baseNmStem=[fNm.stem for fNm in files], 
                             baseNm=[fNm.name for fNm in files], 
                             ext=[fNm.suffix for fNm in files]))

In [23]:
newFiles = build_file_df(rfw.lit_attachment_dir_shared)
print(f'{len(newFiles)=}')
newFiles

len(newFiles)=1651


,fullPath,baseNmStem,baseNm,ext
0,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,1,1.pdf,.pdf
1,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,21ThisPlyometricWorkout,21ThisPlyometricWorkout.html,.html
2,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,2520RecentSeattle,2520RecentSeattle.html,.html
3,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,25WAEngineersLaunched,25WAEngineersLaunched.html,.html
4,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,393d74e0-5451-4aaf-a3ed-717fe55b8346,393d74e0-5451-4aaf-a3ed-717fe55b8346.htm,.htm
...,...,...,...,...
1646,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,Zimmer24thoughtIs10bps,Zimmer24thoughtIs10bps.html,.html
1647,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,zotero-5423,zotero-5423.html,.html
1648,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,Zugno13tradeWindGenMrktQs,Zugno13tradeWindGenMrktQs.pdf,.pdf
1649,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,Zygmunt17fastHyperband,Zygmunt17fastHyperband.html,.html


In [24]:
# oldFiles = [fNm for fNm in old_source_dir.glob('*')]

# oldLUT = pd.DataFrame(dict(fullPath=oldFiles, baseNmStem=[fNm.stem for fNm in oldFiles], ext=[fNm.suffix for fNm in oldFiles])).set_index('baseNmStem')
# oldLUT

oldFiles = build_file_df(old_source_dir)
print(f'{len(oldFiles)=}')
oldFiles

len(oldFiles)=2039


,fullPath,baseNmStem,baseNm,ext
0,c:\Users\scott\OneDrive\share\transfer\papers_...,3tier07windRampSGII,3tier07windRampSGII.pdf,.pdf
1,c:\Users\scott\OneDrive\share\transfer\papers_...,ABB05DakotaTransIncr,ABB05DakotaTransIncr.pdf,.pdf
2,c:\Users\scott\OneDrive\share\transfer\papers_...,ABB05DakotaTransIncr,ABB05DakotaTransIncr.ppt,.ppt
3,c:\Users\scott\OneDrive\share\transfer\papers_...,Abbott11nukeGlbScl,Abbott11nukeGlbScl.pdf,.pdf
4,c:\Users\scott\OneDrive\share\transfer\papers_...,AbdulRahman12flexRampCAISO,AbdulRahman12flexRampCAISO.pdf,.pdf
...,...,...,...,...
2034,c:\Users\scott\OneDrive\share\transfer\papers_...,Zou18multiTaskDisease,Zou18multiTaskDisease.pdf,.pdf
2035,c:\Users\scott\OneDrive\share\transfer\papers_...,Zuber09geneRankCorr,Zuber09geneRankCorr.pdf,.pdf
2036,c:\Users\scott\OneDrive\share\transfer\papers_...,Zuber10varImpSelDecorr,Zuber10varImpSelDecorr.pdf,.pdf
2037,c:\Users\scott\OneDrive\share\transfer\papers_...,Zugno13tradeWindGenMrktQs,Zugno13tradeWindGenMrktQs.pdf,.pdf


In [25]:
oldFilesMatchingDBcitekey = []
files2copy = []
for ix, oldrow in oldFiles.iterrows():
    if oldrow.baseNmStem in zotDBparentCiteKeys:
        oldFilesMatchingDBcitekey.append(oldrow)
        dest_full_path = rfw.lit_attachment_dir_shared / oldrow.baseNm
        if not dest_full_path.exists():
            files2copy.append(dict(oldPath=oldrow.fullPath, destPath=dest_full_path))

            dest_full_path.write_bytes(oldrow.fullPath.read_bytes())

oldFilesMatchingDBcitekey = pd.DataFrame(oldFilesMatchingDBcitekey)
files2copy = pd.DataFrame(files2copy)

print(f'{len(oldFilesMatchingDBcitekey)=}, {len(files2copy)=}')
files2copy

len(oldFilesMatchingDBcitekey)=60, len(files2copy)=0


""


In [26]:
[fNm.name for fNm in files2copy.oldPath]

AttributeError: 'DataFrame' object has no attribute 'oldPath'